In [ ]:
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

import numpy as np
import torch

from loka.models import BreakthroughResNet18, BreakthroughCNN, BreakthroughModel
from loka.data import load_breakthrough_states
from loka.geometrical_methods import herding_order, k_center_order

from tqdm import tqdm

import os
from dotenv import load_dotenv

load_dotenv()

SEED = 42
torch.manual_seed(SEED)

BASE_DIR = os.getenv("BASE_DIR")

data_dir = f"{BASE_DIR}/data/game_states/e.txt"

device = "cuda" if torch.cuda.is_available() else "cpu"

X_np,Y_np = load_breakthrough_states(data_dir)
X,Y = torch.FloatTensor(X_np), torch.FloatTensor(Y_np)

X_3d = X.reshape(-1, 2, 5, 5)                           # (N, 2, 5, 5)
ones = torch.ones(X_3d.shape[0], 1, 5, 5, dtype=X.dtype, device=X.device)
X_3d = torch.cat([X_3d, ones], dim=1)

In [ ]:
import torch
import torch.nn as nn

class ResBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.conv1 = nn.Conv2d(c, c, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(c)
        self.conv2 = nn.Conv2d(c, c, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(c)

    def forward(self, x):
        y = torch.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        return torch.relu(x + y)

class BreakthroughResNet(BreakthroughModel):
    def __init__(self, in_ch=3, channels=32, blocks=2, board=5):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels), nn.ReLU())
        self.tower = nn.Sequential(*[ResBlock(channels) for _ in range(blocks)])
        self.value_head = nn.Sequential(        # AlphaZero value head
            nn.Conv2d(channels, 1, 1, bias=False),
            nn.BatchNorm2d(1), nn.ReLU(),
            nn.Flatten(),
            nn.Linear(board * board, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Tanh())        # drop Tanh if targets aren't in [-1, 1]

    def forward(self, x):
        return self.value_head(self.tower(self.stem(x))).squeeze(-1)

In [ ]:
model = BreakthroughResNet()

In [ ]:
model = BreakthroughCNN()

In [ ]:
# Source - https://stackoverflow.com/a/49201237
# Posted by Fábio Perez, modified by community. See post 'Timeline' for change history
# Retrieved 2026-11-08, License - CC BY-SA 4.0

pytorch_total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
pytorch_total_params

224,161 # Simple CCN
11,177,025 # ResNet18

# Hyper parameters

In [ ]:
epochs = 50
active_learning_rounds = 15

start, end, steps = 0.01, 0.1, 20
SPLITS = np.linspace(start, end, num=steps, endpoint=True)
BATCH_SIZE = 512*2

TOTAL_EPOCHS = {
    'herding' : epochs*active_learning_rounds,
    'k-center' : epochs*active_learning_rounds,
    'forgetting' : epochs*active_learning_rounds,
    'CAL' : epochs,
}

SETTINGS = {
    'model' : BreakthroughResNet,
    'opt' : torch.optim.Adam,
    'lr' : 0.001,
    'loss_fn' : nn.MSELoss
}

herding_order = herding_order(X_np, end, device)
k_center_order = k_center_order(X_np, end,device)

def train(model, optimizer, loss_fn, X, Y, epochs, pbar, pbar_inc):
    model.train()
    n = len(X)
    for _ in range(epochs):
        perm = torch.randperm(n, device=X.device)
        for i in range(0, n, BATCH_SIZE):
            idx = perm[i:i + BATCH_SIZE]
            xb, yb = X[idx], Y[idx]

            optimizer.zero_grad()
            batch_loss = loss_fn(model(xb), yb)
            batch_loss.backward()
            optimizer.step()

        pbar.update(pbar_inc)

# Herding

In [ ]:

total_epochs = TOTAL_EPOCHS['herding']
incs = [int(s * len(X)) for s in SPLITS]
total_work = sum(incs) * total_epochs
pbar = tqdm(total=total_work, desc='Training',
            bar_format="{l_bar}{bar} {percentage:3.0f}% [{elapsed}<{remaining}]")

X_t = X_3d.to(device)
Y_t = Y.to(device)

mse_values = list()
prc_values = list()

for i,s in enumerate(SPLITS):
    torch.manual_seed(SEED)
    model = SETTINGS['model']().to(device)
    optimizer = SETTINGS['opt'](model.parameters(), lr=SETTINGS['lr'])
    loss_fn = SETTINGS['loss_fn']()

    N = int(s*len(X_t))

    train(model, optimizer, loss_fn, X_t[herding_order][:N], Y_t[herding_order][:N], total_epochs, pbar, incs[i])

    with torch.no_grad():
        test_preds = model(X_t)
        mse = loss_fn(test_preds, Y_t).item()
        mse_values.append(mse)
        prc_values.append(s)
        

pbar.close()

In [ ]:
print([n for n, p in model.named_parameters() if p.requires_grad])
# should include 'fc.weight', 'fc.bias'

# K center

# Forgetting

## Forgetting with Herding

## Forgetting with K-Center

# CAL

## Cal with Herding

## Cal with K-center